In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import warnings
import os

import gfdl_utils.core as gu
import CM4Xutils
import cftime
import numpy as np
import pandas as pd
import xarray as xr
import xgcm
import xhistogram
import xwmt

import cmocean
import matplotlib.colors as colors
import matplotlib.ticker as mtick
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib.dates as mdates
plt.rcParams.update({'font.size': 11})

### Load postprocessed model datasets

In [5]:
from common import *
grids = load_datasets()

Inferring Z grid coordinate: depth `z_`
Inferring Z grid coordinate: depth `z_`


#### Compute year in which each grid cell switches from a CFC sink to source

In [22]:
sim = "CM4Xp125_forced"
ds = grids[sim]._ds
ds["fgcfc11_annual"] = ds["fgcfc11"].groupby("time.year").mean()
ds["fgcfc11_annual"] = ds["fgcfc11_annual"].where(ds["deptho"]!=0, np.nan)

always_sink = (ds["fgcfc11_annual"] > -1e-18).rolling({"year":10}, min_periods=1).mean("year").all("year").compute()
source_year = (ds["fgcfc11_annual"].rolling({"year":10}, min_periods=1).mean("year") <= -1e-18).idxmax("year")
source_year = source_year.where((~always_sink) & (ds.deptho!=0))
source_year = xr.where(source_year == 1850, 2100, source_year)

source_year.to_netcdf("../data/processed/cfc11_sink_to_source_transition_year.nc")